#### Use 'elt' kernel / environment. Run shell commands in the parent directory; not the 'notebooks' directory.

#### 1. Create dbt project

```
dbt init olist_dbt
```

In [ ]:
from pathlib import Path
import kaggle

# Configuration
# kaggle_dataset = 'olistbr/brazilian-ecommerce'
# data_dir = Path("./data/olist")
# kaggle_dataset = 'psparks/instacart-market-basket-analysis'
# data_dir = Path("./data/instacart")

data_dir.mkdir(parents=True, exist_ok=True)

# Authenticate with Kaggle
kaggle.api.authenticate()

# Download and extract the dataset
kaggle.api.dataset_download_files(
    kaggle_dataset,
    path=str(data_dir),
    unzip=True
)

#### 2. Upload to BigQuery

In [ ]:
from google.cloud import bigquery

# Configuration
project_id = 'project-7781a5d3-3f4e-4cd5-a43'
bq_dataset = 'brazilian_ecommerce'
# bq_dataset = 'instacart_market_basket_analysis'

# Create BigQuery client
client = bigquery.Client(project=project_id)

dataset_id = f"{project_id}.{bq_dataset}"
# Check if the dataset already exists
try:
    client.get_dataset(dataset_id)
    print(f"Dataset already exists: {dataset_id}")

except Exception:
    dataset = bigquery.Dataset(dataset_id)

    # Set this to the same location you intend to use for your data
    dataset.location = "asia-southeast1"

    dataset = client.create_dataset(dataset)
    print(f"Created dataset: {dataset.full_dataset_id}")

In [ ]:
# Find all CSV files recursively
data_dir = Path("../data/olist")
csv_files = list(data_dir.rglob("*.csv"))
# bypass review file which didn't load properly
#csv_files = list(data_dir.rglob("new_view.csv"))
#csv_files = list(data_dir.rglob("olist_order_reviews_dataset.csv"))
print(csv_files)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

print(f"Found {len(csv_files)} CSV file(s)")

In [ ]:
for csv_file in csv_files:
    # Use the filename as the BigQuery table name
    table_name = csv_file.stem.lower()

    # Replace invalid characters in table names
    table_name = "".join(
        character if character.isalnum() or character == "_" else "_"
        for character in table_name
    )

    table_id = f"{project_id}.{bq_dataset}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        skip_leading_rows=1,
        source_format=bigquery.SourceFormat.CSV,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )

    print(f"Loading {csv_file} into {table_id}")

    with csv_file.open("rb") as source_file:
        load_job = client.load_table_from_file(
            source_file,
            table_id,
            job_config=job_config,
        )

    load_job.result()

    table = client.get_table(table_id)
    print(f"Loaded {table.num_rows} rows into {table_id}")